<a href="https://colab.research.google.com/github/Junhojuno/pytorch-tutorial/blob/master/RNN_seq2seq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Seq2Seq
- 크게 구조는 Encoder와 Decoder로 나뉘고 각각 RNN network를 띄고 있다.
- Encoder에서 입력 문장(input Sequence)을 벡터로 압축해준다.
- Decoder는 이 압축된 벡터를 hidden state로 받아 \<start> 플래그와 함께 cell을 시작한다.
- Decoder에서 output은 다음 sequence의 input으로 들어간다.
- ![Seq2Seq구조](https://)

In [0]:
import random
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

In [0]:
# sample sentences
# tab으로 구분
raw = ["I feel hungry.    나는 배가 고프다.",
       "Pytorch is very easy.    파이토치는 매우 쉽다.",
       "Pytorch is a framwork for deep learning.    파이토치는 딥러닝을 위한 프레임워크이다.",
       "Pytorch is very clear to use.    파이토치는 사용하기에 매우 직관적이다."]

# decoder의 start플래그에 해당하는 토큰과 end플래그에 해당하는 토큰
sos_token = 0 # start of sentences
eos_token = 1 # end of sentences, 문장의 종료를 알려줌

In [0]:

# class for vocabulary related information of data
class Vocab:
    def __init__(self):
        self.vocab2index = {"<SOS>": SOS_token, "<EOS>": EOS_token}
        self.index2vocab = {SOS_token: "<SOS>", EOS_token: "<EOS>"}
        self.vocab_count = {}
        self.n_vocab = len(self.vocab2index)

    def add_vocab(self, sentence):
        for word in sentence.split(" "):
            if word not in self.vocab2index:
                self.vocab2index[word] = self.n_vocab
                self.vocab_count[word] = 1
                self.index2vocab[self.n_vocab] = word
                self.n_vocab += 1
            else:
                self.vocab_count[word] += 1

In [0]:
# filter out the long sentence from source and target data
# 전체 문장의 length에 제한을 걸어준다.
# length 제한 조건에 부합하면 True 아니면 False 반환
def filter_pair(pair, source_max_length, target_max_length):
    return len(pair[0].split(" ")) < source_max_length and len(pair[1].split(" ")) < target_max_length

In [0]:
# Data Preprocessing
# train/test dataset나누고, 길이가 몇으로 할건지 지정 등을 한다.
def preprocess(corpus, source_max_length, target_max_length):
    print("Reading corpus...")
    pairs = []
    for line in corpus:
        pairs.append([s for s in line.strip().lower().split('\t')]) # 예를들면,["I feel hungry.", "나는 배가 고프다."] 이게 1 pair
    print("Read {} sentence pairs".format(len(pairs)))
    
    pairs = [pair for pair in pairs if filter_pair(pair, source_max_length, target_max_length)] # 기준에 부합하는 pair 선별
    print("Trimmed to {} sentence pairs".format(len(pairs)))
    
    # 각 영/한 단어 dictionary
    source_vocab = Vocab() 
    target_vocab = Vocab()
    
    print("Counting words...")
    for pair in pairs:
        source_vocab.add_vocab(pair[0])
        target_vocab.add_vocab(pair[1])
    print("source vocab size =", source_vocab.n_vocab)
    print("target vocab size =", target_vocab.n_vocab)

    return pairs, source_vocab, target_vocab

In [0]:
# Encoder
class Encoder(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(Encoder, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(num_embeddings=input_size, embedding_dim=hidden_size)
        self.gru = nn.GRU(input_size=hidden_size, hidden_size=hidden_size)
        
    def forward(self, x, hidden):
        x = self.embedding(x).view(1,1,-1)
        x, hidden = gru(x, hidden) # 두개가 argument로 들어가는데....왜 두개지? 한개면 안되나?
        return x, hidden        

In [0]:
# Decoder
class Decoder(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(Decoder, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size)
        self.out = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)
        
    def forward(self, x, hidden):
        x = self.embedding(x).view(1,1,-1) # one hot encoding된 단어벡터가 embedding matrix를 만나 차원이 줄어든 input이 된다.
        x, hidden = self.gru(x, hidden)
        x = self.softmax(self.out(x[0]))
        return x, hidden

In [0]:
# convert sentence to the index tensor
# sentence를 one hot encoding시키고 tensor의 형태로 바꿔주는 역할 (sentence --> one hot vector)
def tensorize(vocab, sentence):
    indexes = [vocab.vocab2index[word] for word in sentence.split(" ")]
    indexes.append(vocab.vocab2index["<EOS>"])
    return torch.Tensor(indexes).long().to(device).view(-1, 1)